# [실습 04] ReAct 루프 직접 구현하기

> **연계**: 제2부 04장(ReAct) · **환경**: Google Colab · **모델**: 오픈웨이트 `Qwen/Qwen2.5-1.5B-Instruct`

**학습 목표**
- **Thought → Action → Observation** 루프를 직접 코드로 구현한다.
- 도구(검색) 결과를 관찰해 다음 추론에 반영하는 과정을 체험한다.

In [ ]:
!pip install -q transformers accelerate torch

## 1. 간단한 지식 검색 도구

In [ ]:
KB = {
    "에펠탑 설계자": "귀스타브 에펠",
    "귀스타브 에펠 출생연도": "1832년",
}
def search(query):
    for k, v in KB.items():
        if k in query or query in k:
            return v
    return "검색 결과 없음"
print(search("에펠탑 설계자"))

## 2. ReAct 루프

모델에게 `Thought/Action/Answer` 형식을 유도하고, `Action`이 나오면 검색을 실행해 `Observation`으로 되돌려 줍니다.

In [ ]:
import torch, re
from transformers import pipeline
gen = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
               torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
               device_map="auto")

SYS = '''질문에 답하기 위해 아래 형식을 반복하라.
Thought: 생각
Action: search("검색어")   ← 도구가 필요할 때
정답을 알면: Answer: 최종답
한 번에 한 줄만 출력하라.'''

def react(question, max_steps=4):
    scratch = f"Question: {question}\n"
    for _ in range(max_steps):
        msg = [{"role": "system", "content": SYS},
               {"role": "user", "content": scratch}]
        line = gen(msg, max_new_tokens=60, do_sample=False)[0]["generated_text"][-1]["content"].strip().splitlines()[0]
        print(line)
        scratch += line + "\n"
        if line.startswith("Answer:"):
            return line
        m = re.search(r'search\("?(.+?)"?\)', line)
        if m:
            obs = search(m.group(1))
            print("Observation:", obs)
            scratch += f"Observation: {obs}\n"
    return "(최대 단계 도달)"

react("에펠탑을 지은 사람이 태어난 해는?")

## 3. 정리
- **Thought(추론) → Action(도구) → Observation(관찰)** 루프를 직접 구현했다.
- 관찰이 다시 추론을 교정하는 것이 ReAct의 핵심이다(04-1).
- **더 해보기**: `KB`에 새 사실을 넣고 2단계 이상 추론이 필요한 질문을 던져 보세요.